# Phase 2: Phát triển Core Model (Kỹ thuật Chuyên Sâu)

Notebook này triển khai toàn bộ **Hybrid Sybil Detection Model** bao gồm:
- 🔄 **Random Walk Algorithm**: Phát hiện hành vi xâm nhập trên đồ thị cấu trúc
- 📊 **Transition Matrix P**: Ma trận chuyển xác suất và phân phối dừng (Stationary Distribution)
- 📝 **LDA + LSTM**: Trích xuất tương quan ngữ nghĩa từ nội dung đánh giá
- 🎯 **Hybrid Model**: Kết hợp SNA (60%) + NLP (40%) để tính toán Hybrid Score
- 💾 **Export Artifacts**: Lưu model data, scores, và sybil candidates cho Phase 3 & 4

**Mục tiêu**: Xác định top-20 Sybil accounts với độ chính xác 87±5%

In [ ]:
import subprocess
import sys

# Install required packages
packages = [
    'networkx',
    'scikit-learn',
    'gensim',
    'tensorflow',
    'pandas',
    'numpy',
    'scipy',
    'matplotlib',
    'seaborn',
    'pyarrow'
]

for pkg in packages:
    try:
        __import__(pkg.replace('-', '_'))
        print(f"✓ {pkg} already installed")
    except ImportError:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("\n✓ All packages installed")

✓ networkx already installed
Installing scikit-learn...


You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
/Users/mac/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import pickle
from pathlib import Path
from collections import defaultdict, Counter
import json

In [ ]:
# ML & NLP libraries - Optimized Import
print("📦 Loading ML/NLP libraries (this may take 10-15 seconds)...")
import time
start_time = time.time()

# Fast imports first
try:
    from sklearn.preprocessing import StandardScaler
    print(f"  ✓ StandardScaler loaded ({time.time()-start_time:.1f}s)")
except ImportError as e:
    print(f"  ⚠️ StandardScaler import failed: {e}")

# Defer heavy imports with lazy loading
class LazyImport:
    """Lazy load heavy libraries on first use"""
    _lda = None
    _vectorizer = None
    _tf = None
    _keras = None
    
    @classmethod
    def get_lda(cls):
        if cls._lda is None:
            print("  ⏳ Loading LDA (first time use)...")
            from sklearn.decomposition import LatentDirichletAllocation
            cls._lda = LatentDirichletAllocation
        return cls._lda
    
    @classmethod
    def get_vectorizer(cls):
        if cls._vectorizer is None:
            print("  ⏳ Loading CountVectorizer...")
            from sklearn.feature_extraction.text import CountVectorizer
            cls._vectorizer = CountVectorizer
        return cls._vectorizer
    
    @classmethod
    def get_tf(cls):
        if cls._tf is None:
            print("  ⏳ Loading TensorFlow (this is slow, please wait...)...")
            import tensorflow as tf
            cls._tf = tf
        return cls._tf
    
    @classmethod
    def get_keras(cls):
        if cls._keras is None:
            print("  ⏳ Loading Keras layers...")
            from tensorflow import keras
            cls._keras = keras
        return cls._keras

# Create shortcuts for immediate use
from scipy.sparse import csr_matrix

print(f"✓ ML/NLP imports optimized (ready in {time.time()-start_time:.1f}s)")
print("⏳ Heavy imports (TensorFlow/Keras) will load on first use\n")

📦 Loading ML/NLP libraries (this may take 10-15 seconds)...
  ✓ StandardScaler loaded (1.5s)
✓ ML/NLP imports optimized (ready in 1.5s)
⏳ Heavy imports (TensorFlow/Keras) will load on first use



In [ ]:

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

In [ ]:

# ============================================================================
# GPU SETUP FOR MACBOOK (Metal Performance Shaders)
# ============================================================================
print("🔧 GPU Setup for MacBook Pro...")

# Check available devices
try:
    # TensorFlow - use Metal GPU
    physical_devices = tf.config.list_physical_devices('GPU')
    if physical_devices:
        print(f"✓ TensorFlow GPU: {physical_devices}")
        tf.config.experimental.set_memory_growth(physical_devices[0], True)
    else:
        print("⚠️ No GPU found for TensorFlow, using CPU")
        physical_devices = tf.config.list_physical_devices('CPU')
        print(f"✓ Using CPU: {physical_devices}")
except:
    print("⚠️ TensorFlow GPU setup failed, continuing with CPU")

# Config
PROJECT_DIR = Path("/Users/mac/Downloads/MXH FINAL")
DATA_DIR = PROJECT_DIR / "data"
GOLD_DIR = DATA_DIR / "gold"
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print("✓ Import libraries completed")
print(f"✓ TensorFlow version: {tf.__version__}")
print(f"✓ Project directory: {PROJECT_DIR}")

In [ ]:
# ============================================================================
# SECTION 1: RANDOM WALK ALGORITHM & TRANSITION MATRIX
# ============================================================================

class RandomWalkAnomalyDetector:
    """
    Phát hiện hành vi xâm nhập sử dụng Random Walk trên đồ thị tương tác user-product.
    Tính toán ma trận chuyển xác suất P và phân phối dừng (stationary distribution)
    để xác định các tài khoản nghi vấn.
    """
    
    def __init__(self, graph=None):
        self.graph = graph or nx.DiGraph()
        self.transition_matrix = None
        self.stationary_dist = None
        self.suspicion_scores = {}
        
    def build_transition_matrix(self, damping_factor=0.85, max_iterations=100, tolerance=1e-6):
        """
        Xây dựng ma trận chuyển xác suất P từ cấu trúc đồ thị.
        Sử dụng PageRank-like approach để tính toán.
        """
        print("🔧 Xây dựng ma trận chuyển xác suất...")
        
        n_nodes = self.graph.number_of_nodes()
        nodes_list = list(self.graph.nodes())
        node_to_idx = {node: i for i, node in enumerate(nodes_list)}
        
        # Khởi tạo ma trận chuyển xác suất
        P = np.zeros((n_nodes, n_nodes))
        
        for node in nodes_list:
            out_edges = list(self.graph.out_edges(node, data=True))
            out_degree = len(out_edges)
            
            if out_degree > 0:
                for _, target, data in out_edges:
                    weight = data.get('weight', 1.0)
                    P[node_to_idx[node], node_to_idx[target]] = weight / sum([d.get('weight', 1.0) for _, _, d in out_edges])
            else:
                # Nếu không có outgoing edges, phân bổ đều cho tất cả nodes
                P[node_to_idx[node], :] = 1.0 / n_nodes
        
        # Tính toán stationary distribution sử dụng Power Iteration
        print("  → Tính toán phân phối dừng (Stationary Distribution)...")
        x = np.ones(n_nodes) / n_nodes
        
        for iteration in range(max_iterations):
            x_new = damping_factor * (P.T @ x) + (1 - damping_factor) / n_nodes
            
            if np.linalg.norm(x_new - x) < tolerance:
                print(f"  ✓ Hội tụ sau {iteration} lần lặp")
                break
            x = x_new
        
        self.transition_matrix = P
        self.stationary_dist = x
        self.node_to_idx = node_to_idx
        self.idx_to_node = {i: node for node, i in node_to_idx.items()}
        
        print("✓ Ma trận chuyển xác suất hoàn tất")
        return P, x
    
    def calculate_suspicion_scores(self, method='pagerank_anomaly'):
        """
        Tính toán chỉ số nghi vấn dựa trên Random Walk results.
        Các phương pháp:
        - pagerank_anomaly: Độ lệch từ expected stationary distribution
        - degree_anomaly: Bất thường về degree
        - clustering_anomaly: Bất thường về clustering coefficient
        """
        print(f"🎯 Tính toán chỉ số nghi vấn ({method})...")
        
        scores = {}
        
        if method == 'pagerank_anomaly':
            # Tính PageRank scores
            pagerank_scores = nx.pagerank(self.graph, alpha=0.85)
            mean_score = np.mean(list(pagerank_scores.values()))
            std_score = np.std(list(pagerank_scores.values()))
            
            for node, score in pagerank_scores.items():
                # Z-score: (score - mean) / std
                anomaly = abs(score - mean_score) / (std_score + 1e-6)
                scores[node] = anomaly
                
        elif method == 'degree_anomaly':
            # Bất thường về in-degree và out-degree
            in_degrees = dict(self.graph.in_degree())
            out_degrees = dict(self.graph.out_degree())
            
            mean_in = np.mean(list(in_degrees.values()))
            std_in = np.std(list(in_degrees.values()))
            mean_out = np.mean(list(out_degrees.values()))
            std_out = np.std(list(out_degrees.values()))
            
            for node in self.graph.nodes():
                z_in = abs(in_degrees[node] - mean_in) / (std_in + 1e-6)
                z_out = abs(out_degrees[node] - mean_out) / (std_out + 1e-6)
                scores[node] = (z_in + z_out) / 2
                
        elif method == 'clustering_anomaly':
            # Bất thường về clustering coefficient
            clustering_coeff = nx.clustering(self.graph.to_undirected())
            values = list(clustering_coeff.values())
            mean_cluster = np.mean(values)
            std_cluster = np.std(values)
            
            for node, coeff in clustering_coeff.items():
                anomaly = abs(coeff - mean_cluster) / (std_cluster + 1e-6)
                scores[node] = anomaly
        
        self.suspicion_scores = scores
        print("✓ Tính toán chỉ số nghi vấn hoàn tất")
        return scores
    
    def get_top_suspicious_accounts(self, top_k=20):
        """Trả về top-k tài khoản nghi vấn nhất."""
        sorted_scores = sorted(self.suspicion_scores.items(), 
                             key=lambda x: x[1], reverse=True)
        return sorted_scores[:top_k]


print("✓ Class RandomWalkAnomalyDetector được định nghĩa")

In [ ]:
# ============================================================================
# SECTION 2: LDA + LSTM FOR SEMANTIC CORRELATION EXTRACTION
# ============================================================================

class SemanticCorrelationExtractor:
    """
    Trích xuất tương quan ngữ nghĩa từ nội dung đánh giá sử dụng:
    - LDA: Phát hiện các chủ đề tiềm ẩn
    - LSTM: Mã hóa các chuỗi từ để phát hiện các mẫu ngôn ngữ bất thường
    """
    
    def __init__(self, max_features=5000, lda_topics=10):
        self.max_features = max_features
        self.lda_topics = lda_topics
        self.lda_model = None
        self.vectorizer = None
        self.lstm_model = None
        self.topic_distributions = {}
        
    def prepare_text_data(self, texts, min_df=5, max_df=0.8):
        """Chuẩn bị dữ liệu văn bản cho LDA."""
        print("📝 Chuẩn bị dữ liệu văn bản...")
        
        self.vectorizer = CountVectorizer(
            max_features=self.max_features,
            min_df=min_df,
            max_df=max_df,
            stop_words='english'
        )
        
        doc_term_matrix = self.vectorizer.fit_transform(texts)
        print(f"  ✓ Tạo ma trận từ-tài liệu: {doc_term_matrix.shape}")
        
        return doc_term_matrix
    
    def fit_lda_model(self, doc_term_matrix, n_components=None, max_iter=20):
        """Huấn luyện mô hình LDA."""
        print("🧠 Huấn luyện mô hình LDA...")
        
        n_components = n_components or self.lda_topics
        
        self.lda_model = LatentDirichletAllocation(
            n_components=n_components,
            max_iter=max_iter,
            learning_method='online',
            random_state=42,
            n_jobs=-1
        )
        
        self.lda_model.fit(doc_term_matrix)
        
        # Lấy topic distribution cho mỗi tài liệu
        topic_dist = self.lda_model.transform(doc_term_matrix)
        print(f"  ✓ LDA hoàn tất: {n_components} chủ đề")
        
        return topic_dist
    
    def build_lstm_text_encoder(self, vocab_size=5000, embedding_dim=128, max_length=100):
        """Xây dựng LSTM encoder cho phân tích chuỗi văn bản."""
        print("🔨 Xây dựng LSTM encoder...")
        
        model = keras.Sequential([
            Embedding(vocab_size, embedding_dim, input_length=max_length),
            Bidirectional(LSTM(64, return_sequences=True)),
            Dropout(0.3),
            Bidirectional(LSTM(32)),
            Dropout(0.3),
            Dense(16, activation='relu'),
            Dense(8, activation='sigmoid')  # Output semantic features
        ])
        
        model.compile(optimizer='adam', loss='mse')
        self.lstm_model = model
        
        print(f"  ✓ LSTM model tạo thành công")
        self.max_length = max_length
        self.vocab_size = vocab_size
        
        return model
    
    def extract_semantic_features(self, texts, lda_weight=0.6, lstm_weight=0.4):
        """Trích xuất tổng hợp các đặc trưng ngữ nghĩa từ LDA và LSTM."""
        print("✨ Trích xuất các đặc trưng ngữ nghĩa...")
        
        # LDA features
        doc_term_matrix = self.prepare_text_data(texts)
        lda_features = self.fit_lda_model(doc_term_matrix)
        
        # Chuẩn hóa LDA features
        lda_features_norm = (lda_features - lda_features.mean(axis=0)) / (lda_features.std(axis=0) + 1e-6)
        
        semantic_features = {
            'lda': lda_features_norm,
            'lda_dominant_topics': np.argmax(lda_features, axis=1),
            'lda_topic_entropy': -np.sum(lda_features * np.log(lda_features + 1e-6), axis=1),
        }
        
        print("✓ Trích xuất đặc trưng ngữ nghĩa hoàn tất")
        
        return semantic_features


print("✓ Class SemanticCorrelationExtractor được định nghĩa")

In [ ]:
# ============================================================================
# SECTION 3: HYBRID MODEL INTEGRATION & HYBRID SCORE COMPUTATION
# ============================================================================

class HybridSybilDetectionModel:
    """
    Mô hình hybrid kết hợp Social Network Analysis (SNA) và NLP.
    Sử dụng tương quan ngữ nghĩa để điều chỉnh trọng số cạnh đồ thị.
    """
    
    def __init__(self, graph, text_data):
        self.graph = graph
        self.text_data = text_data
        self.random_walk_detector = RandomWalkAnomalyDetector(graph)
        self.semantic_extractor = SemanticCorrelationExtractor()
        self.hybrid_scores = {}
        self.edge_weights_adjusted = {}
        
    def adjust_edge_weights_by_semantic_correlation(self, semantic_features, 
                                                    semantic_anomaly_threshold=0.7):
        """
        Điều chỉnh trọng số cạnh dựa trên tương quan ngữ nghĩa.
        Các cạnh với semantic anomaly cao sẽ có trọng số giảm.
        """
        print("🔀 Điều chỉnh trọng số cạnh bằng tương quan ngữ nghĩa...")
        
        # Tính toán semantic anomaly score cho từng tài khoản (node)
        lda_entropy = semantic_features['lda_topic_entropy']
        lda_entropy_norm = (lda_entropy - lda_entropy.mean()) / (lda_entropy.std() + 1e-6)
        semantic_anomaly = 1 / (1 + np.exp(-lda_entropy_norm))  # Sigmoid normalization
        
        node_list = list(self.graph.nodes())
        node_to_semantic_anomaly = {node: score for node, score in zip(node_list, semantic_anomaly[:len(node_list)])}
        
        # Điều chỉnh trọng số cạnh
        for u, v, data in self.graph.edges(data=True):
            original_weight = data.get('weight', 1.0)
            
            # Nếu node nguồn có semantic anomaly cao, giảm trọng số
            anomaly_u = node_to_semantic_anomaly.get(u, 0.5)
            anomaly_v = node_to_semantic_anomaly.get(v, 0.5)
            
            # Hệ số điều chỉnh: nếu anomaly cao, weight giảm
            adjustment_factor = 1.0 - (anomaly_u + anomaly_v) / 2 * 0.3
            adjusted_weight = original_weight * adjustment_factor
            
            self.edge_weights_adjusted[(u, v)] = {
                'original': original_weight,
                'adjusted': adjusted_weight,
                'anomaly_u': anomaly_u,
                'anomaly_v': anomaly_v
            }
            
            # Cập nhật trọng số trong graph
            self.graph[u][v]['weight'] = adjusted_weight
        
        print(f"✓ Điều chỉnh trọng số cho {len(self.edge_weights_adjusted)} cạnh")
        
        return node_to_semantic_anomaly
    
    def compute_hybrid_scores(self, sna_weight=0.6, nlp_weight=0.4):
        """
        Tính toán Hybrid Score kết hợp:
        - SNA Score: từ Random Walk & Suspicion Index
        - NLP Score: từ Semantic Anomaly & Topic Consistency
        """
        print("🎪 Tính toán Hybrid Scores...")
        
        # Bước 1: Tính Random Walk suspicion scores
        self.random_walk_detector.build_transition_matrix()
        sna_scores = self.random_walk_detector.calculate_suspicion_scores(method='pagerank_anomaly')
        
        # Chuẩn hóa SNA scores về [0, 1]
        sna_values = np.array(list(sna_scores.values()))
        sna_scores_norm = (sna_values - sna_values.min()) / (sna_values.max() - sna_values.min() + 1e-6)
        sna_scores_norm = {node: score for node, score in 
                          zip(sna_scores.keys(), sna_scores_norm)}
        
        # Bước 2: Trích xuất semantic features
        texts = [self.text_data.get(node, "") for node in self.graph.nodes()]
        semantic_features = self.semantic_extractor.extract_semantic_features(texts)
        node_to_semantic_anomaly = self.adjust_edge_weights_by_semantic_correlation(semantic_features)
        
        # Bước 3: Tính toán Hybrid Scores
        hybrid_scores = {}
        for node in self.graph.nodes():
            sna_score = sna_scores_norm.get(node, 0.5)
            nlp_score = node_to_semantic_anomaly.get(node, 0.5)
            
            # Hybrid score: weighted combination
            hybrid_score = sna_weight * sna_score + nlp_weight * nlp_score
            hybrid_scores[node] = {
                'hybrid_score': hybrid_score,
                'sna_score': sna_score,
                'nlp_score': nlp_score,
                'combined_rank': 0  # Sẽ tính sau
            }
        
        # Ranking
        sorted_nodes = sorted(hybrid_scores.items(), 
                            key=lambda x: x[1]['hybrid_score'], reverse=True)
        for rank, (node, score_dict) in enumerate(sorted_nodes, 1):
            hybrid_scores[node]['combined_rank'] = rank
        
        self.hybrid_scores = hybrid_scores
        print(f"✓ Tính toán Hybrid Scores cho {len(hybrid_scores)} nodes")
        
        return hybrid_scores
    
    def get_top_sybil_candidates(self, top_k=50, threshold=0.7):
        """Lấy top-k tài khoản Sybil candidate với threshold."""
        candidates = [(node, scores['hybrid_score'], scores['combined_rank']) 
                     for node, scores in self.hybrid_scores.items()
                     if scores['hybrid_score'] >= threshold]
        
        candidates.sort(key=lambda x: x[1], reverse=True)
        return candidates[:top_k]
    
    def export_model_data(self, output_path):
        """Xuất dữ liệu mô hình cho các giai đoạn tiếp theo."""
        export_data = {
            'hybrid_scores': self.hybrid_scores,
            'edge_weights_adjusted': self.edge_weights_adjusted,
            'random_walk_detector': self.random_walk_detector,
            'semantic_extractor': self.semantic_extractor
        }
        
        with open(output_path, 'wb') as f:
            pickle.dump(export_data, f)
        
        print(f"✓ Dữ liệu mô hình xuất tại: {output_path}")
        
        return export_data


print("✓ Class HybridSybilDetectionModel được định nghĩa")

In [ ]:
# ============================================================================
# SECTION 4: LOAD DATA & BUILD INITIAL GRAPH
# ============================================================================

# Tải dữ liệu từ Preprocessing phase
print("📊 Tải dữ liệu từ Preprocessing phase...")

# Tìm file dữ liệu
data_files = list(GOLD_DIR.glob("*.json"))
print(f"Found {len(data_files)} data files")

if not data_files:
    print("⚠️ Không tìm thấy file dữ liệu. Tạo dữ liệu mẫu...")
    
    # Tạo dữ liệu mẫu cho demo
    sample_data = []
    for i in range(100):
        sample_data.append({
            'user_id': f'user_{i}',
            'product_id': f'prod_{i % 20}',
            'rating': np.random.randint(1, 6),
            'text': f'This is review {i} with some sample text content for testing.',
            'timestamp': i
        })
    
    data = sample_data
else:
    # Load từ file
    with open(data_files[0], 'r') as f:
        data = json.load(f) if isinstance(f.read(), str) else []
        f.seek(0)
        data = json.load(f)

print(f"✓ Loaded {len(data)} records")
df = pd.DataFrame(data)
print(df.head())

In [ ]:
# ============================================================================
# SECTION 5: CONSTRUCT USER-PRODUCT INTERACTION GRAPH
# ============================================================================

print("🔗 Xây dựng User-Product Interaction Graph...")

# Khởi tạo graph
G = nx.DiGraph()

# Thêm các nodes và edges từ dữ liệu
user_review_count = defaultdict(int)
product_review_count = defaultdict(int)

for idx, row in df.iterrows():
    user = row['user_id']
    product = row['product_id']
    
    # Tạo node cho user và product
    G.add_node(user, node_type='user')
    G.add_node(product, node_type='product')
    
    # Tạo edge từ user đến product với trọng số là rating
    weight = row.get('rating', 1) / 5.0  # Chuẩn hóa rating
    
    if G.has_edge(user, product):
        G[user][product]['weight'] += weight
        G[user][product]['count'] += 1
    else:
        G.add_edge(user, product, weight=weight, count=1)
    
    user_review_count[user] += 1
    product_review_count[product] += 1

print(f"✓ Graph statistics:")
print(f"  - Nodes: {G.number_of_nodes()}")
print(f"  - Edges: {G.number_of_edges()}")
print(f"  - Users: {len([n for n, d in G.nodes(data=True) if d.get('node_type') == 'user'])}")
print(f"  - Products: {len([n for n, d in G.nodes(data=True) if d.get('node_type') == 'product'])}")

# Thêm thông tin review text vào node
user_reviews = defaultdict(list)
for idx, row in df.iterrows():
    user = row['user_id']
    user_reviews[user].append(row.get('text', ''))

# Tạo mapping text cho từng user
text_data = {user: ' '.join(reviews) for user, reviews in user_reviews.items()}

In [ ]:
# ============================================================================
# SECTION 6: RUN HYBRID MODEL DETECTION
# ============================================================================

print("\n" + "="*60)
print("🚀 RUNNING HYBRID SYBIL DETECTION MODEL")
print("="*60)

# Khởi tạo Hybrid Model
hybrid_model = HybridSybilDetectionModel(G, text_data)

# Tính toán Hybrid Scores
hybrid_scores = hybrid_model.compute_hybrid_scores(sna_weight=0.6, nlp_weight=0.4)

# Lấy top-k Sybil candidates
top_sybils = hybrid_model.get_top_sybil_candidates(top_k=20, threshold=0.5)

print("\n📋 TOP-20 SYBIL CANDIDATES:")
print("-" * 60)
for rank, (node, score, combined_rank) in enumerate(top_sybils, 1):
    score_details = hybrid_scores[node]
    print(f"{rank:2d}. {node:15s} | Hybrid: {score:.4f} | SNA: {score_details['sna_score']:.4f} | NLP: {score_details['nlp_score']:.4f}")

print("\n✓ Hybrid Model Detection completed")

In [ ]:
# ============================================================================
# SECTION 7: EXPORT MODEL ARTIFACTS
# ============================================================================

# Ensure all dependencies are initialized
from pathlib import Path
import pandas as pd

# Define OUTPUT_DIR if not already defined
if 'OUTPUT_DIR' not in globals():
    PROJECT_DIR = Path("/Users/mac/Downloads/MXH FINAL")
    OUTPUT_DIR = PROJECT_DIR / "outputs"
    OUTPUT_DIR.mkdir(exist_ok=True)
    print("⚠️ WARNING: Initializing OUTPUT_DIR. Make sure to run setup cells first!")
    print(f"✓ OUTPUT_DIR set to: {OUTPUT_DIR}")

# Verify hybrid_model and top_sybils exist
if 'hybrid_model' not in globals() or 'top_sybils' not in globals():
    print("\n❌ ERROR: hybrid_model or top_sybils not defined!")
    print("Please run SECTION 6 (Run Hybrid Model Detection) first!")
    print("\nTo fix:")
    print("1. Go to top of notebook (Ctrl+Home)")
    print("2. Run all cells in order (Shift+Enter repeatedly)")
    raise NameError("Missing variables: hybrid_model or top_sybils. Run previous sections first!")

print("\n💾 Xuất Model Artifacts...")

# Tạo thư mục output
model_output_dir = OUTPUT_DIR / "Phase2_CoreModel"
model_output_dir.mkdir(exist_ok=True)

# Xuất hybrid model data
model_data_path = model_output_dir / "hybrid_model_data.pkl"
hybrid_model.export_model_data(model_data_path)

# Xuất hybrid scores thành CSV
hybrid_scores_df = pd.DataFrame([
    {
        'node': node,
        'hybrid_score': scores['hybrid_score'],
        'sna_score': scores['sna_score'],
        'nlp_score': scores['nlp_score'],
        'combined_rank': scores['combined_rank']
    }
    for node, scores in hybrid_scores.items()
]).sort_values('hybrid_score', ascending=False)

hybrid_scores_path = model_output_dir / "hybrid_scores.csv"
hybrid_scores_df.to_csv(hybrid_scores_path, index=False)
print(f"✓ Hybrid scores saved: {hybrid_scores_path}")

# Xuất top sybils
sybil_candidates_df = pd.DataFrame([
    {'rank': rank, 'node': node, 'score': score}
    for rank, (node, score, _) in enumerate(top_sybils, 1)
])

sybil_path = model_output_dir / "sybil_candidates.csv"
sybil_candidates_df.to_csv(sybil_path, index=False)
print(f"✓ Sybil candidates saved: {sybil_path}")

# Xuất adjusted edge weights
edge_weights_df = pd.DataFrame([
    {
        'source': u,
        'target': v,
        'original_weight': data['original'],
        'adjusted_weight': data['adjusted'],
        'anomaly_source': data['anomaly_u'],
        'anomaly_target': data['anomaly_v']
    }
    for (u, v), data in hybrid_model.edge_weights_adjusted.items()
])

edge_weights_path = model_output_dir / "edge_weights_adjusted.csv"
edge_weights_df.to_csv(edge_weights_path, index=False)
print(f"✓ Edge weights saved: {edge_weights_path}")

print("\n✓ Phase 2 - Core Model Development hoàn tất!")